In [345]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import math
import numpy as np
import matplotlib.pyplot as plt
from engine import Value
from nn import Neuron, Layer, MLP


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [344]:
x = [2.0, 3.0, -1.0] #inputs
n = MLP(3, [4,4,1])
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [-1.0, 0.5, 1.0],
    [0.5, 1.0, 2.0],
]

ys = [1.0, -1.0, -1.0, 1.0] #desired outputs

In [339]:
for k in range(20):

    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)** 2 for ygt, yout in zip(ys, ypred))

    for p in n.parameters():
        p.grad = 0.0

    loss.backward()

    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)

0 1.5575475480120446e-12
1 1.557547547734929e-12
2 1.557547547734929e-12
3 1.557547547734929e-12
4 1.557547547734929e-12
5 1.557547547734929e-12
6 1.557547547734929e-12
7 1.557547547734929e-12
8 1.5575475474578136e-12
9 1.5575475474578136e-12
10 1.5575475474578136e-12
11 1.5575475469035824e-12
12 1.5575475469035824e-12
13 1.5575475469035824e-12
14 1.557547546626467e-12
15 1.557547546626467e-12
16 1.557547546626467e-12
17 1.557547546626467e-12
18 1.557547546626467e-12
19 1.557547546626467e-12


In [342]:
ypred

[Value(data=0.9999999999927278),
 Value(data=-0.9999999999951825),
 Value(data=-0.9999987519825536),
 Value(data=0.9999999999919843)]

In [343]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot